In [1]:
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(r"d:\ML Project - E-commerce product search with semantic ranking")
DATA = ROOT / "data"
sys.path.append(str(ROOT))

from src.evaluate import evaluate_run, build_run_dict

corpus = pd.read_parquet(DATA / "corpus.parquet")
queries = pd.read_parquet(DATA / "queries.parquet")
qrels = pd.read_parquet(DATA / "qrels.parquet")
hybrid_run = pd.read_parquet(DATA / "run_hybrid.parquet")

text_of = dict(zip(corpus["product_id"], corpus["search_text"]))
title_of = dict(zip(corpus["product_id"], corpus["product_title"]))
hy_d = build_run_dict(hybrid_run)

print("Corpus:", len(corpus), "| Queries:", len(queries), "| Hybrid rows:", len(hybrid_run))

Corpus: 40063 | Queries: 500 | Hybrid rows: 50000


In [2]:
from sentence_transformers import CrossEncoder

ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=192, device="cpu")
print("Loaded!")

d:\ML Project - E-commerce product search with semantic ranking\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\ML Project - E-commerce product search with semantic ranking\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vansh\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activa

Loaded!


In [3]:
pairs = [
    ("something to keep my coffee hot", "Insulated stainless steel travel mug, keeps drinks hot for 12 hours"),
    ("something to keep my coffee hot", "Coffee table book about world cuisine"),
    ("something to keep my coffee hot", "Hot pink yoga mat"),
]
scores = ce.predict(pairs)
for (q, d), s in zip(pairs, scores):
    print(f"{s:7.2f}  {d}")

  -5.06  Insulated stainless steel travel mug, keeps drinks hot for 12 hours
  -9.75  Coffee table book about world cuisine
 -10.61  Hot pink yoga mat


In [4]:
def rerank_query(qtext, candidate_ids, top_n=10):
    pairs = [(qtext, text_of[p]) for p in candidate_ids]
    scores = ce.predict(pairs, batch_size=32)
    order = np.argsort(scores)[::-1][:top_n]
    return [(candidate_ids[i], float(scores[i])) for i in order]

row = queries.iloc[0]
qid, qtext = row["query_id"], row["query"]
candidates = hy_d[qid][:50]
labels = qrels[qrels["query_id"] == qid].set_index("product_id")["esci_label"]

print("QUERY:", qtext, "\n")
print("HYBRID top 5:")
for p in candidates[:5]:
    print(f"  [{labels.get(p, 'not judged'):>10}] {title_of[p][:75]}")

print("\nRE-RANKED top 5:")
for p, s in rerank_query(qtext, candidates, 5):
    print(f"  [{labels.get(p, 'not judged'):>10}] {title_of[p][:75]}")

QUERY: inflatable dildo 

HYBRID top 5:
  [         E] Trinity Vibes Inflatable Suction Cup Dildo
  [         S] 10" Monster Silicone Realistic Dildo with Dual Density Anal Dildo Huge Suct
  [not judged] Dildo Elastic D'ildo for Adullt Women's Body Stimulation and Relaxation Mǎs
  [not judged] Jasonwell Inflatable Dinosaur Pool Float for Boys Girls Adults 124'' Giant 
  [not judged] Inflatable Bonkin' Sheep

RE-RANKED top 5:
  [         E] Trinity Vibes Inflatable Suction Cup Dildo
  [         E] G Spot Vibrator Inflatable Massager, 2020 New Vibrators Sex Toy for Vaginal
  [         S] 10" Monster Silicone Realistic Dildo with Dual Density Anal Dildo Huge Suct
  [not judged] Dildo Elastic D'ildo for Adullt Women's Body Stimulation and Relaxation Mǎs
  [not judged] Clitoral Sucking Vibrator,G Spot Vibrating Realistic Dildo 2 in 1 Vibrators


In [5]:
start = time.time()
for qtext, qid in zip(queries["query"].head(5), queries["query_id"].head(5)):
    rerank_query(qtext, hy_d[qid][:50], 10)
per_query = (time.time() - start) / 5

print(f"Time per query (50 candidates): {per_query*1000:.0f} ms")
print(f"Estimated time for 500 queries: {per_query*500/60:.1f} minutes")

Time per query (50 candidates): 3508 ms
Estimated time for 500 queries: 29.2 minutes


In [6]:
DEPTH = 50
qids = queries["query_id"].tolist()
qtexts = dict(zip(queries["query_id"], queries["query"]))

all_pairs, owners = [], []
for qid in qids:
    for p in hy_d[qid][:DEPTH]:
        all_pairs.append((qtexts[qid], text_of[p]))
        owners.append((qid, p))

print("Total pairs:", len(all_pairs))

start = time.time()
all_scores = ce.predict(all_pairs, batch_size=64, show_progress_bar=True)
print(f"Done in {(time.time() - start)/60:.1f} minutes")

Total pairs: 25000


Batches: 100%|██████████| 391/391 [09:43<00:00,  1.49s/it]

Done in 9.7 minutes


In [7]:
raw = pd.DataFrame(owners, columns=["query_id", "product_id"])
raw["score"] = all_scores
raw.to_parquet(DATA / "ce_scores_raw.parquet", index=False)

rerank_run = raw.sort_values(["query_id", "score"], ascending=[True, False]).copy()
rerank_run["rank"] = rerank_run.groupby("query_id").cumcount() + 1
rerank_run.to_parquet(DATA / "run_rerank.parquet", index=False)
print("Saved!")

Saved!


In [8]:
bm25_run = pd.read_parquet(DATA / "run_bm25.parquet")
sem_run = pd.read_parquet(DATA / "run_semantic.parquet")

for name, run in [("BM25", bm25_run), ("Semantic", sem_run),
                  ("Hybrid", hybrid_run), ("Hybrid + Re-ranker", rerank_run)]:
    print(f"{name:20}", evaluate_run(run, qrels, k=10))

BM25                 {'Recall@10': 0.3362, 'MRR@10': 0.7332, 'NDCG@10': 0.4952, 'queries_evaluated': 500}
Semantic             {'Recall@10': 0.336, 'MRR@10': 0.7456, 'NDCG@10': 0.4986, 'queries_evaluated': 500}
Hybrid               {'Recall@10': 0.3673, 'MRR@10': 0.7583, 'NDCG@10': 0.5335, 'queries_evaluated': 500}
Hybrid + Re-ranker   {'Recall@10': 0.4045, 'MRR@10': 0.8103, 'NDCG@10': 0.5953, 'queries_evaluated': 500}


In [9]:
rr_scores = evaluate_run(rerank_run, qrels, k=10)

results = pd.read_csv(DATA / "results.csv")
results = results[results["Method"] != "Hybrid + Re-ranker"]
results = pd.concat([results, pd.DataFrame([{"Method": "Hybrid + Re-ranker", **rr_scores}])],
                    ignore_index=True)
results.to_csv(DATA / "results.csv", index=False)
results

,Method,Recall@10,MRR@10,NDCG@10,queries_evaluated
0,BM25,0.3362,0.7332,0.4952,500
1,Semantic,0.3360,0.7456,0.4986,500
2,Hybrid,0.3673,0.7583,0.5335,500
3,Hybrid + Re-ranker,0.4045,0.8103,0.5953,500


In [10]:
from src.evaluate import build_qrels_dict, dcg

qrels_d = build_qrels_dict(qrels)
rr_d = build_run_dict(rerank_run)

def ndcg_q(run_d, qid, k=10):
    judged = qrels_d[qid]
    gains = [judged.get(p, 0) for p in run_d.get(qid, [])[:k]]
    ideal = dcg(sorted(judged.values(), reverse=True)[:k])
    return dcg(gains) / ideal if ideal > 0 else 0.0

diff = pd.DataFrame({
    "query_id": list(qrels_d.keys()),
    "hybrid": [ndcg_q(hy_d, q) for q in qrels_d],
    "rerank": [ndcg_q(rr_d, q) for q in qrels_d],
})
diff["gain"] = diff["rerank"] - diff["hybrid"]
diff = diff.merge(queries, on="query_id")

print("Re-ranker better:", (diff["gain"] > 0.05).sum())
print("Hybrid better:   ", (diff["gain"] < -0.05).sum())
print("About equal:     ", (diff["gain"].abs() <= 0.05).sum())

print("\nRe-ranker much better:")
print(diff.sort_values("gain", ascending=False).head(6)[["query", "hybrid", "rerank"]].to_string())
print("\nHybrid much better:")
print(diff.sort_values("gain").head(6)[["query", "hybrid", "rerank"]].to_string())

Re-ranker better: 218
Hybrid better:    88
About equal:      194

Re-ranker much better:
                                   query    hybrid    rerank
26                           40 cake pan  0.000000  0.776367
167  entertainment stand for tv up to 90  0.430677  1.000000
225                              hinatab  0.211450  0.749722
370     rib protector under armour youth  0.079907  0.617702
61                  automatic wine glass  0.250288  0.774181
341                  playstation 2 games  0.144598  0.649558

Hybrid much better:
                                     query    hybrid    rerank
102                                  brick  0.919721  0.386853
85                 blocked christmas words  0.613147  0.177239
13                           10 in tablets  0.637152  0.241607
389                   screen doors roll up  0.697026  0.343406
185                              fly argic  0.301030  0.000000
9    0sxk ofrece fidget toys not expensive  0.788954  0.500973
